In [6]:
import pandas as pd 

In [127]:
def create_clean_df_for_Content_pipeline(movies_df,tags_df):
    new_movies_df = movies_df[['movieId','title','genres']].copy()
    # cleaning movies_df
    new_movies_df['title'] = (
        movies_df['title'].fillna("")
        .astype(str)
        .str.strip()
    )
    new_movies_df['genres'] = (
        new_movies_df['genres'].fillna("")
        .astype(str)
        .str.replace("|"," ")
        .str.strip()
    )
    
    # cleaning tags_df
    new_tags_df = tags_df[["movieId","tag"]].copy()
    new_tags_df.dropna(subset=['movieId','tag'])
    
    new_tags_df['tag'] = (
    new_tags_df['tag']
    .astype(str)
    .str.strip()
    .str.lower()
    )
    
    new_tags_df = new_tags_df[new_tags_df['tag'] != ""]
    
    tags_grouped = (
    new_tags_df.groupby('movieId')['tag']
    .apply(lambda crr_mov_tags : " ".join(crr_mov_tags)) # this lambda function input tags of first movie and join them with " " and then moves to next movie tags and do same upto so on
    .reset_index(name='tags') # it cut out the movieId from the resultant series into separate column and the name="tags" changes the name of remaining tag  series to tagss
)
    
    # merging the tags and movies_df
    movies_df_final = pd.merge(
        new_movies_df,
        tags_grouped,
        on='movieId',
        how='left',
        validate='one_to_one' # validating so that each movieId appears one in both data frames otherwise it will raise error 
    )
    
    # cleaning final frame 
    movies_df_final["tags"] = movies_df_final['tags'].fillna("")
    movies_df_final = movies_df_final.drop_duplicates(subset=["movieId"])
    movies_df_final = movies_df_final.reset_index(drop=True)
    
    return movies_df_final


In [73]:
movies = pd.read_csv(r'../datasets/movies.csv')
tags = pd.read_csv(r"../datasets/tags.csv")
tmdb = pd.read_csv(r'../datasets/tmdb_movies.csv')
ratings = pd.read_csv(r'../datasets/ratings.csv')

# Cleaning Tmdb dataframe

In [41]:
tmdb

,movieId,tmdbId,overview,release_date,poster_path,backdrop_path
0,1,862,"Led by Woody, Andy's toys live happily in his ...",1995-11-22,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg
1,2,8844,When siblings Judy and Peter discover an encha...,1995-12-15,/bdHG5Mo83VPobeZZdlSz0Y7HQHB.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg
2,3,15602,A family wedding reignites the ancient feud be...,1995-12-22,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,/1o4vuCHpmd4DXofMYDUwpnhKiuy.jpg
3,4,31357,"Cheated on, mistreated and stepped on, the wom...",1995-12-22,/4wjGMwPsdlvi025ZqR4rXnFDvBz.jpg,/jZjoEKXMTDoZAGdkjhAdJaKtXSN.jpg
4,5,11862,Just when George Banks has recovered from his ...,1995-12-08,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,/lEsjVrGU21BeJjF5AF9EWsihDpw.jpg
...,...,...,...,...,...,...
9615,193581,432131,"Ciel learns of a ""Aurora Society"", that is rum...",2017-01-21,/4jU2Bdk1MB2OvP4BBvTJIPQc0BF.jpg,/clFQR5zwzDrayMVVe7axo0YLFfh.jpg
9616,193583,445030,Six thousand years before Sora and Shiro were ...,2017-07-15,/cCBB6BGRj5nCTaEgogDtkHfjOLK.jpg,/zmJtqKaByFqYrxipeXyoAUSTD1w.jpg
9617,193585,479308,A woman deals with the toxic water scandal in ...,2017-10-28,/iPzkjNWpK7ud5pE997eGaB4QhaG.jpg,/tcM6XoTYKATJyNTIuGPk4yh0g6N.jpg
9618,193587,483455,A large scale catastrophe is occurring across ...,2018-03-03,/gCfN6xWPyxphtgtLuDxT5s0eSoq.jpg,/vjnS4iu0SdwXm2LLqZGNCfpId9t.jpg


In [ ]:
# droping rows that have duplicate or None values 
tmdb.drop_duplicates(subset=["movieId"],inplace=True)
tmdb.drop_duplicates(subset=["overview"],inplace=True)
tmdb.drop_duplicates(subset=["poster_path"],inplace=True)
tmdb.drop_duplicates(subset=["backdrop_path"],inplace=True)
tmdb.drop_duplicates(subset=["movieId"],inplace=True)
tmdb.dropna(subset=["backdrop_path"],inplace=True)

In [124]:
tmdb_for_merge = tmdb[["movieId","overview"]] .copy()

# cleaning ratings data frame to get top popular movies

In [76]:
# Getting number of ratings applied to each movie and avg_rating applied to each movie by users
popular_movies = ratings.groupby('movieId').agg(
    num_ratings = ('rating','count'),
    avg_rating = ('rating','mean')
).reset_index().copy()

In [92]:
merged_pop_mov = pd.merge(popular_movies,movies,on="movieId",how="inner")

In [93]:
merged_pop_mov

,movieId,num_ratings,avg_rating,title,genres,year,imdbId,tmdbId
0,1,215,3.920930,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,114709,862
1,2,110,3.431818,Jumanji (1995),Adventure|Children|Fantasy,1995,113497,8844
2,3,52,3.259615,Grumpier Old Men (1995),Comedy|Romance,1995,113228,15602
3,4,7,2.357143,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,114885,31357
4,5,49,3.071429,Father of the Bride Part II (1995),Comedy,1995,113041,11862
...,...,...,...,...,...,...,...,...
9699,193581,1,4.000000,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,2017,5476944,432131
9700,193583,1,3.500000,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,2017,5914996,445030
9701,193585,1,3.500000,Flint (2017),Drama,2017,6397426,479308
9702,193587,1,3.500000,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,2018,8391976,483455


In [108]:
new_popular_movies = merged_pop_mov.query('num_ratings>90 and year>1998 and avg_rating>2.8').copy()

In [109]:
new_popular_movies

,movieId,num_ratings,avg_rating,title,genres,year,imdbId,tmdbId
1880,2502,94,4.090426,Office Space (1999),Comedy|Crime,1999,151804,1542
1936,2571,278,4.192446,"Matrix, The (1999)",Action|Sci-Fi|Thriller,1999,133093,603
1976,2628,140,3.107143,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi,1999,120915,1893
2011,2683,121,3.198347,Austin Powers: The Spy Who Shagged Me (1999),Action|Adventure|Comedy,1999,145660,817
2028,2706,103,3.378641,American Pie (1999),Comedy|Romance,1999,163651,2105
2075,2762,179,3.893855,"Sixth Sense, The (1999)",Drama|Horror|Mystery,1999,167404,745
2141,2858,204,4.056373,American Beauty (1999),Drama|Romance,1999,169547,14
2221,2959,218,4.272936,Fight Club (1999),Action|Crime|Drama|Thriller,1999,137523,550
2254,2997,99,3.954545,Being John Malkovich (1999),Comedy|Drama|Fantasy,1999,120601,492
2350,3114,97,3.860825,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1999,120363,863


In [110]:
new_popular_movies.drop(columns=['tmdbId'],inplace=True)

In [116]:
final_popular_df = pd.merge(new_popular_movies,tmdb,on='movieId',how="left")

In [117]:
final_popular_df.drop(columns=['tmdbId','year'],inplace=True)

In [120]:
final_popular_df.to_csv('../datasets/new_data_sets/top_movies.csv')

# merging cleaned merged tags and movies data frame with tmdbId frame 

In [128]:
new_df = create_clean_df_for_Content_pipeline(movies,tags)

In [134]:
new_df

,movieId,title,genres,tags
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure Children Fantasy,fantasy magic board game robin williams game
2,3,Grumpier Old Men (1995),Comedy Romance,moldy old
3,4,Waiting to Exhale (1995),Comedy Drama Romance,
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake
...,...,...,...,...
9717,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy,
9718,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy,
9719,193585,Flint (2017),Drama,
9720,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation,


In [132]:
content_filt_df = pd.merge(new_df,tmdb_for_merge,on='movieId',how='inner')

In [142]:
content_filt_df.dropna(subset=['overview'],inplace=True)

In [145]:
content_filt_df

,movieId,title,genres,tags,overview
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar pixar fun,"Led by Woody, Andy's toys live happily in his ..."
1,2,Jumanji (1995),Adventure Children Fantasy,fantasy magic board game robin williams game,When siblings Judy and Peter discover an encha...
2,3,Grumpier Old Men (1995),Comedy Romance,moldy old,A family wedding reignites the ancient feud be...
3,4,Waiting to Exhale (1995),Comedy Drama Romance,,"Cheated on, mistreated and stepped on, the wom..."
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,Just when George Banks has recovered from his ...
...,...,...,...,...,...
9607,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy,,"Ciel learns of a ""Aurora Society"", that is rum..."
9608,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy,,Six thousand years before Sora and Shiro were ...
9609,193585,Flint (2017),Drama,,A woman deals with the toxic water scandal in ...
9610,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation,,A large scale catastrophe is occurring across ...


In [ ]:
content_filt_df['content'] = (
    content_filt_df['title']+" "
    +content_filt_df['genres'] + " "
    +content_filt_df['tags'] + " "
    + content_filt_df['overview']
).str.strip()

In [150]:
content_filt_df

,movieId,title,genres,tags,overview,content
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar pixar fun,"Led by Woody, Andy's toys live happily in his ...",Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure Children Fantasy,fantasy magic board game robin williams game,When siblings Judy and Peter discover an encha...,Jumanji (1995) Adventure Children Fantasy fant...
2,3,Grumpier Old Men (1995),Comedy Romance,moldy old,A family wedding reignites the ancient feud be...,Grumpier Old Men (1995) Comedy Romance moldy o...
3,4,Waiting to Exhale (1995),Comedy Drama Romance,,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale (1995) Comedy Drama Romance ...
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,Just when George Banks has recovered from his ...,Father of the Bride Part II (1995) Comedy preg...
...,...,...,...,...,...,...
9607,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy,,"Ciel learns of a ""Aurora Society"", that is rum...",Black Butler: Book of the Atlantic (2017) Acti...
9608,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy,,Six thousand years before Sora and Shiro were ...,No Game No Life: Zero (2017) Animation Comedy ...
9609,193585,Flint (2017),Drama,,A woman deals with the toxic water scandal in ...,Flint (2017) Drama A woman deals with the tox...
9610,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation,,A large scale catastrophe is occurring across ...,Bungo Stray Dogs: Dead Apple (2018) Action Ani...


In [166]:
content_filt_df.to_csv('../datasets/new_data_sets/frame_of_colab_filt.csv')

## preparing a full movie info data set so that I use it to display information on website

In [153]:
movies.isnull().sum()

movieId    0
title      0
genres     0
year       0
imdbId     0
tmdbId     0
dtype: int64

In [161]:
info_frame = pd.merge(movies,tmdb,on='movieId',how='left')

In [164]:
final_info_df = info_frame.drop(columns=['year','tmdbId_x','tmdbId_y',]).copy()

In [163]:
final_info_df

,movieId,title,genres,imdbId,overview,release_date,poster_path,backdrop_path
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,"Led by Woody, Andy's toys live happily in his ...",1995-11-22,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,When siblings Judy and Peter discover an encha...,1995-12-15,/bdHG5Mo83VPobeZZdlSz0Y7HQHB.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,A family wedding reignites the ancient feud be...,1995-12-22,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,/1o4vuCHpmd4DXofMYDUwpnhKiuy.jpg
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,"Cheated on, mistreated and stepped on, the wom...",1995-12-22,/4wjGMwPsdlvi025ZqR4rXnFDvBz.jpg,/jZjoEKXMTDoZAGdkjhAdJaKtXSN.jpg
4,5,Father of the Bride Part II (1995),Comedy,113041,Just when George Banks has recovered from his ...,1995-12-08,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,/lEsjVrGU21BeJjF5AF9EWsihDpw.jpg
...,...,...,...,...,...,...,...,...
9717,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,5476944,"Ciel learns of a ""Aurora Society"", that is rum...",2017-01-21,/4jU2Bdk1MB2OvP4BBvTJIPQc0BF.jpg,/clFQR5zwzDrayMVVe7axo0YLFfh.jpg
9718,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,5914996,Six thousand years before Sora and Shiro were ...,2017-07-15,/cCBB6BGRj5nCTaEgogDtkHfjOLK.jpg,/zmJtqKaByFqYrxipeXyoAUSTD1w.jpg
9719,193585,Flint (2017),Drama,6397426,A woman deals with the toxic water scandal in ...,2017-10-28,/iPzkjNWpK7ud5pE997eGaB4QhaG.jpg,/tcM6XoTYKATJyNTIuGPk4yh0g6N.jpg
9720,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,8391976,A large scale catastrophe is occurring across ...,2018-03-03,/gCfN6xWPyxphtgtLuDxT5s0eSoq.jpg,/vjnS4iu0SdwXm2LLqZGNCfpId9t.jpg


In [165]:
final_info_df.to_csv('../datasets/new_data_sets/movie_info.csv')